# Mengukur Jarak Antar Data

Mengukur jarak (dissimilarity) antar dua objek data dari dataset Titanic berdasarkan tipe datanya masing-masing.

In [1]:
import pandas as pd
import numpy as np
from scipy.spatial.distance import jaccard, euclidean

df = pd.read_csv('../assets/train.csv')
df.head(5)

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


---
## 1. Binary

Menghitung jarak data binary beberapa sampel dari data di atas.

Dari data di atas, fitur dengan tipe data binary adalah `Survived`. Fitur ini hanya memiliki dua nilai yaitu `0` (tidak selamat) dan `1` (selamat).

Karena `Survived` merupakan **asymmetric binary** (nilai 1 lebih bermakna), maka digunakan **Jaccard Distance**:

$$d(i,j) = \frac{r + s}{q + r + s}$$

- **q** = keduanya bernilai 1  
- **r** = i = 0, j = 1  
- **s** = i = 1, j = 0

In [2]:
binary_cols = ['Survived']

p1 = df[binary_cols].iloc[0]
p2 = df[binary_cols].iloc[1]

print(f"Row 1 - Survived: {p1['Survived']}")
print(f"Row 2 - Survived: {p2['Survived']}")
print()

jaccard_distance = jaccard(p1, p2)
print(f"Jaccard Distance: {jaccard_distance:.4f}")

Row 1 - Survived: 0
Row 2 - Survived: 1

Jaccard Distance: 1.0000


> **Catatan:** Pada implementasi di atas, data yang digunakan adalah baris pertama dan kedua. Row 1 tidak selamat (0) dan Row 2 selamat (1), sehingga jaraknya bernilai **1.0** (maksimum / paling berbeda).

---
## 2. Nominal (Kategorikal)

Menghitung jarak data nominal beberapa sampel dari data di atas.

Pada data di atas, yang dihitung hanya atribut dengan tipe data nominal, selain itu diabaikan.

Atribut yang dihitung adalah `Sex` dan `Embarked`, menggunakan **Simple Matching**:

$$d(i,j) = \frac{p - m}{p}$$

- **p** = jumlah atribut nominal  
- **m** = jumlah atribut yang nilainya sama

In [4]:
nominal_cols = ['Sex', 'Embarked']

p1 = df.loc[0, nominal_cols].values
p2 = df.loc[1, nominal_cols].values

print(f"Row 1 → Sex: {p1[0]}, Embarked: {p1[1]}")
print(f"Row 2 → Sex: {p2[0]}, Embarked: {p2[1]}")
print()

distance = np.mean(p1 != p2)
print(f"Simple Matching Distance: {distance:.4f}")

Row 1 → Sex: male, Embarked: S
Row 2 → Sex: female, Embarked: C

Simple Matching Distance: 1.0000


> **Catatan:** Pada implementasi di atas, data yang digunakan adalah baris pertama dan kedua. Sex berbeda (male vs female) dan Embarked berbeda (S vs C), sehingga jaraknya bernilai **1.0**.

---
## 3. Ordinal

Menghitung jarak data ordinal beberapa sampel dari data di atas.

Fitur dengan tipe data ordinal adalah `Pclass` (kelas penumpang: 1, 2, 3). Nilainya memiliki **urutan/peringkat** namun jarak antar kelas tidak tentu sama.

Langkah perhitungan:
1. Gunakan ranking yang sudah ada (1, 2, 3)
2. Normalisasi ke skala [0,1]: $z_{if} = \dfrac{r_{if} - 1}{M_f - 1}$, dengan $M_f = 3$
3. Hitung selisih: $d = |z_i - z_j|$

In [5]:
Mf = 3  # jumlah rank: kelas 1, 2, 3

r1 = df.loc[0, 'Pclass']
r2 = df.loc[1, 'Pclass']

z1 = (r1 - 1) / (Mf - 1)
z2 = (r2 - 1) / (Mf - 1)

distance = abs(z1 - z2)

print(f"Row 1 → Pclass = {r1} → z = ({r1}-1)/(3-1) = {z1:.4f}")
print(f"Row 2 → Pclass = {r2} → z = ({r2}-1)/(3-1) = {z2:.4f}")
print()
print(f"Ordinal Distance = |{z1:.4f} - {z2:.4f}| = {distance:.4f}")

Row 1 → Pclass = 3 → z = (3-1)/(3-1) = 1.0000
Row 2 → Pclass = 1 → z = (1-1)/(3-1) = 0.0000

Ordinal Distance = |1.0000 - 0.0000| = 1.0000


> **Catatan:** Pada implementasi di atas, data yang digunakan adalah baris pertama dan kedua. Pclass Row 1 = 3 (kelas bawah) dan Pclass Row 2 = 1 (kelas atas), sehingga setelah dinormalisasi jaraknya bernilai **1.0** (paling berbeda).

---
## 4. Numerik

Menghitung jarak data numerik beberapa sampel dari data di atas.

Fitur dengan tipe data numerik adalah `Age` dan `Fare`. Sebelum dihitung jaraknya, data dinormalisasi terlebih dahulu menggunakan **Z-score**:

$$z = \frac{x - \mu}{\sigma}$$

Kemudian dihitung **Euclidean Distance**:

$$d(i,j) = \sqrt{\sum_{f=1}^{p}(z_{if} - z_{jf})^2}$$

In [6]:
numeric_cols = ['Age', 'Fare']

mean = df[numeric_cols].mean()
std  = df[numeric_cols].std()

p1_z = (df.loc[0, numeric_cols] - mean) / std
p2_z = (df.loc[1, numeric_cols] - mean) / std

print("Statistik dataset:")
for col in numeric_cols:
    print(f"  {col}: mean={mean[col]:.2f}, std={std[col]:.2f}")

print()
print("Nilai setelah Z-score normalisasi:")
for col in numeric_cols:
    print(f"  Row 1 → {col}: {df.loc[0, col]} → z = {p1_z[col]:.4f}")
    print(f"  Row 2 → {col}: {df.loc[1, col]} → z = {p2_z[col]:.4f}")

print()
distance = euclidean(p1_z, p2_z)
print(f"Euclidean Distance: {distance:.4f}")

Statistik dataset:
  Age: mean=29.70, std=14.53
  Fare: mean=32.20, std=49.69

Nilai setelah Z-score normalisasi:
  Row 1 → Age: 22.0 → z = -0.5300
  Row 2 → Age: 38.0 → z = 0.5714
  Row 1 → Fare: 7.25 → z = -0.5022
  Row 2 → Fare: 71.2833 → z = 0.7864

Euclidean Distance: 1.6952


> **Catatan:** Pada implementasi di atas, data yang digunakan adalah baris pertama dan kedua. Perbedaan Age (22 vs 38) dan Fare (7.25 vs 71.28) yang cukup besar menghasilkan jarak Euclidean yang besar setelah dinormalisasi.

---
## 5. Campuran (Mixed)

Menggabungkan seluruh perhitungan jarak di atas menjadi satu nilai jarak untuk data dengan **tipe campuran**.

$$d(i,j) = \frac{\sum_{f=1}^{p} \delta_{ij}^{(f)} \cdot d_{ij}^{(f)}}{\sum_{f=1}^{p} \delta_{ij}^{(f)}}$$

- $\delta = 1$ jika atribut **tersedia** (tidak missing)  
- $\delta = 0$ jika atribut **kosong/missing** → di-skip

Setiap jarak per tipe data yang sudah dihitung di atas dijumlahkan lalu dirata-ratakan.

In [7]:
row1 = df.iloc[0]
row2 = df.iloc[1]

# Binary: Survived (Jaccard)
d_binary = jaccard(row1[['Survived']], row2[['Survived']])

# Nominal: Sex & Embarked (Simple Matching)
n_cols = ['Sex', 'Embarked']
d_nominal = np.mean(df.loc[0, n_cols].values != df.loc[1, n_cols].values)

# Ordinal: Pclass
Mf = 3
z1 = (row1['Pclass'] - 1) / (Mf - 1)
z2 = (row2['Pclass'] - 1) / (Mf - 1)
d_ordinal = abs(z1 - z2)

# Numerik: Age & Fare (Z-score + Euclidean)
num_cols = ['Age', 'Fare']
mean = df[num_cols].mean()
std  = df[num_cols].std()
p1_z = (df.loc[0, num_cols] - mean) / std
p2_z = (df.loc[1, num_cols] - mean) / std
d_numeric = euclidean(p1_z, p2_z)

# Gabungkan
jarak = {
    'Binary  (Survived)'      : d_binary,
    'Nominal (Sex, Embarked)' : d_nominal,
    'Ordinal (Pclass)'        : d_ordinal,
    'Numerik (Age, Fare)'     : d_numeric,
}
delta = {k: 1 for k in jarak}

print(f"{'Tipe Data':<30} {'delta':>6} {'Jarak':>10}")
print("-" * 50)
for k, v in jarak.items():
    print(f"{k:<30} {delta[k]:>6}   {v:>10.4f}")

numerator   = sum(delta[k] * jarak[k] for k in jarak)
denominator = sum(delta[k] for k in jarak)
d_campuran  = numerator / denominator

print("-" * 50)
print(f"\nSigma (delta x d) = {numerator:.4f}")
print(f"Sigma delta       = {denominator}")
print(f"\nJarak Campuran Row 1 vs Row 2 = {numerator:.4f} / {denominator} = {d_campuran:.4f}")

Tipe Data                       delta      Jarak
--------------------------------------------------
Binary  (Survived)                  1       1.0000
Nominal (Sex, Embarked)             1       1.0000
Ordinal (Pclass)                    1       1.0000
Numerik (Age, Fare)                 1       1.6952
--------------------------------------------------

Sigma (delta x d) = 4.6952
Sigma delta       = 4

Jarak Campuran Row 1 vs Row 2 = 4.6952 / 4 = 1.1738


> **Catatan:** Pada implementasi di atas, jarak campuran dihitung dengan merata-ratakan seluruh jarak per tipe data. Hasilnya mencerminkan seberapa berbeda Row 1 (laki-laki, kelas 3, tidak selamat) dengan Row 2 (perempuan, kelas 1, selamat) secara keseluruhan.